**This Jupyter Notebook (`live_object_detection.ipynb`) demonstrates live object detection using a pre-trained YOLO (You Only Look Once) model**
                                                                                                                
- It captures video from your webcam, processes each frame with the YOLO model, and displays the results with bounding boxes and class labels overlaid on the video feed.

-----------------

In [1]:
pip install ultralytics

  Using cached opencv_python-4.11.0.86-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached torch-2.6.0-cp312-cp312-win_amd64.whl.metadata (28 kB)
  Using cached torchvision-0.21.0-cp312-cp312-win_amd64.whl.metadata (6.3 kB)
  Using cached ultralytics_thop-2.0.14-py3-none-any.whl.metadata (9.4 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/915.2 kB ? eta -:--:--
   --------------------------------------- 915.2/915.2 kB 13.9 MB/s eta 0:00:00
Using cached opencv_python-4.11.0.86-cp37-abi3-win_amd64.whl (39.5 MB)
Using cached torch-2.6.0-cp312-cp312-win_amd64.whl (204.1 MB)
Using cached sympy-1.13.1-py3-none-any.whl (6.2 MB)
Using cached torchvision-0.21.0-cp312-cp312-win_amd64.whl (1.6 MB)
Using cached ultralytics_thop-2.0.14-py3-none-any.whl (26 kB)
  Attempting uninstall: sympy
    Found existing installation: sympy 1.13.3
    Uninstalling sympy-1.13.3:
      Successfully uninstalled sympy-1.13.3
Note: you may need t

In [7]:
from ultralytics import YOLO

In [ ]:
from ultralytics import YOLO
import cv2
import torch

# start webcam
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Could not open webcam (index 0). Is a camera connected and free?")
cap.set(3, 640)
cap.set(4, 480)

# model
model = YOLO("yolo-Weights/yolo11s.pt")

# only show detections at or above this confidence (raise to suppress bad guesses)
CONF_THRESHOLD = 0.5

# run on GPU if available, otherwise CPU
device = 0 if torch.cuda.is_available() else "cpu"

# object classes (COCO labels, provided by the model)
classNames = model.names

try:
    while True:
        success, img = cap.read()
        if not success:
            break
        results = model(img, device=device, conf=CONF_THRESHOLD, verbose=False)

        # coordinates
        for r in results:
            boxes = r.boxes

            for box in boxes:
                # bounding box
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)  # convert to int values

                # put box in cam
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 255), 3)

                # class name and confidence
                cls = int(box.cls[0])
                conf = float(box.conf[0])
                label = f"{classNames[cls]} {conf:.2f}"

                # object details
                org = [x1, y1]
                font = cv2.FONT_HERSHEY_SIMPLEX
                fontScale = 1
                color = (255, 0, 0)
                thickness = 2

                cv2.putText(img, label, org, font, fontScale, color, thickness)

        cv2.imshow('Webcam', img)
        if (cv2.waitKey(1) & 0xFF) == ord('q'):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()